# ana_pll full workflow

Run this notebook from top to bottom to generate or reuse the matrix-analysis grids and display the PNG figures directly in the notebook.

Sections:

- `step1`: lifted system with observables `[x, y, x^2]`.
- `step2`: lifted system with observables `[x, y, y^2]` and the default `Sigma` coupling.
- `step2s`: a second `step2` run with a separate `Sigma` coupling/output directory. By default this contrasts with `step2`, and all parameters are controlled below.
- `kuramoto`: clustered Kuramoto experiment with order-parameter traces and `A^T A` spectra/vectors.


## Parameters

Change only this cell for the common workflow controls. Set `FORCE_RERUN=True` when you want to regenerate figures even if matching outputs already exist.

In [ ]:
from pathlib import Path
import fnmatch
import sys

import pandas as pd
from IPython.display import Image, Markdown, display

# Run switches.
RUN_STEP1 = True
RUN_STEP2 = True
RUN_STEP2S = True
RUN_KURAMOTO = True

# If False, the notebook reuses complete existing outputs and only displays them.
FORCE_RERUN = False

# Image display controls. Use None to display every matching PNG.
DISPLAY_LIMIT = None
DISPLAY_WIDTH = 1200

# Optional display filters. Examples: ["case1*.png", "case2*.png"].
STEP1_DISPLAY_PATTERNS = ["*.png"]
STEP2_DISPLAY_PATTERNS = ["*.png"]
STEP2S_DISPLAY_PATTERNS = ["*.png"]
KURAMOTO_DISPLAY_PATTERNS = ["*.png"]

# Step2 Sigma coupling choices are: "y_y2", "x_y2", "x_y".
STEP2_SIGMA_COUPLING = "y_y2"
STEP2S_SIGMA_COUPLING = "x_y2"

cwd = Path.cwd().resolve()
repo_root = next(
    candidate
    for candidate in (cwd, *cwd.parents)
    if (candidate / "tools.py").exists()
)
module_dir = repo_root / "exp" / "analysitic"
if str(module_dir) not in sys.path:
    sys.path.append(str(module_dir))

from plot_ana_pll_backward_svd_grid import (
    REFERENCE_LAM_MU_PAIRS as STEP1_DEFAULT_REFERENCE_LAM_MU_PAIRS,
    DEFAULT_A_VALUES as STEP1_DEFAULT_A_VALUES,
    DEFAULT_B_RATIOS as STEP1_DEFAULT_B_RATIOS,
    DEFAULT_MU_SWEEPS as STEP1_DEFAULT_MU_SWEEPS,
    DEFAULT_LAM_SWEEPS as STEP1_DEFAULT_LAM_SWEEPS,
)
from plot_ana_pll2_backward_svd_grid import (
    REFERENCE_RHO_KAPPA_PAIRS as STEP2_DEFAULT_REFERENCE_RHO_KAPPA_PAIRS,
    DEFAULT_SIGMA_A_VALUES as STEP2_DEFAULT_SIGMA_A_VALUES,
    DEFAULT_SIGMA_B_RATIOS as STEP2_DEFAULT_SIGMA_B_RATIOS,
    DEFAULT_KAPPA_SWEEPS as STEP2_DEFAULT_KAPPA_SWEEPS,
    DEFAULT_RHO_SWEEPS as STEP2_DEFAULT_RHO_SWEEPS,
)
from plot_ana_pllkur_kuramoto_ata_grid import (
    KuramotoConfig,
    DEFAULT_CASE1_FIXED_K_INTRA,
    DEFAULT_CASE1_K_INTER_VALUES,
    DEFAULT_CASE2_FIXED_K_INTER,
    DEFAULT_CASE2_K_INTRA_VALUES,
)

STEP1_OUTPUT_DIR = module_dir / "figs" / "ana_pll"
STEP2_OUTPUT_DIR = module_dir / "figs" / "ana_pll2"
STEP2S_OUTPUT_DIR = module_dir / "figs" / "ana_pll2s"
KURAMOTO_OUTPUT_DIR = module_dir / "figs" / "ana_pllkur"

# Step1 parameter grids.
STEP1_REFERENCE_LAM_MU_PAIRS = STEP1_DEFAULT_REFERENCE_LAM_MU_PAIRS
STEP1_A_VALUES = STEP1_DEFAULT_A_VALUES
STEP1_B_RATIOS = STEP1_DEFAULT_B_RATIOS
STEP1_MU_SWEEPS = STEP1_DEFAULT_MU_SWEEPS
STEP1_LAM_SWEEPS = STEP1_DEFAULT_LAM_SWEEPS

# Step2 parameter grids.
STEP2_REFERENCE_RHO_KAPPA_PAIRS = STEP2_DEFAULT_REFERENCE_RHO_KAPPA_PAIRS
STEP2_SIGMA_A_VALUES = STEP2_DEFAULT_SIGMA_A_VALUES
STEP2_SIGMA_B_RATIOS = STEP2_DEFAULT_SIGMA_B_RATIOS
STEP2_KAPPA_SWEEPS = STEP2_DEFAULT_KAPPA_SWEEPS
STEP2_RHO_SWEEPS = STEP2_DEFAULT_RHO_SWEEPS

# Step2s can use a separate grid, or reuse step2's grid as below.
STEP2S_REFERENCE_RHO_KAPPA_PAIRS = STEP2_REFERENCE_RHO_KAPPA_PAIRS
STEP2S_SIGMA_A_VALUES = STEP2_SIGMA_A_VALUES
STEP2S_SIGMA_B_RATIOS = STEP2_SIGMA_B_RATIOS
STEP2S_KAPPA_SWEEPS = STEP2_KAPPA_SWEEPS
STEP2S_RHO_SWEEPS = STEP2_RHO_SWEEPS

# Kuramoto parameter grids.
KURAMOTO_CONFIG = KuramotoConfig()
KURAMOTO_CASE1_FIXED_K_INTRA = DEFAULT_CASE1_FIXED_K_INTRA
KURAMOTO_CASE1_K_INTER_VALUES = DEFAULT_CASE1_K_INTER_VALUES
KURAMOTO_CASE2_FIXED_K_INTER = DEFAULT_CASE2_FIXED_K_INTER
KURAMOTO_CASE2_K_INTRA_VALUES = DEFAULT_CASE2_K_INTRA_VALUES

repo_root, module_dir

## Shared Helpers

In [ ]:
if str(module_dir) not in sys.path:
    sys.path.append(str(module_dir))

from plot_ana_pll_backward_svd_grid import (
    make_experiment_specs as make_step1_specs,
    run_experiments as run_step1_experiments,
)
from plot_ana_pll2_backward_svd_grid import (
    make_experiment_specs as make_step2_specs,
    run_experiments as run_step2_experiments,
)
from plot_ana_pllkur_kuramoto_ata_grid import (
    make_experiment_specs as make_kuramoto_specs,
    run_experiments as run_kuramoto_experiments,
)


def specs_table(section, specs):
    return pd.DataFrame(
        [
            {
                "section": section,
                "figure": spec.key,
                "n_combos": len(spec.combos),
                "plot_evd": getattr(spec, "plot_evd", None),
                "share_phase_limits": getattr(spec, "share_phase_limits", None),
            }
            for spec in specs
        ]
    )


def output_is_ready(output_dir, expected_sigma_coupling=None):
    summary_path = output_dir / "case_summary.csv"
    if not summary_path.exists():
        return False, "missing case_summary.csv"

    try:
        summary = pd.read_csv(summary_path)
    except Exception as exc:
        return False, f"cannot read case_summary.csv: {exc}"

    if summary.empty:
        return False, "case_summary.csv is empty"
    if "figure" not in summary.columns:
        return False, "case_summary.csv has no figure column"

    png_stems = {path.stem for path in output_dir.glob("*.png")}
    figures = set(summary["figure"].dropna().astype(str).unique())
    missing_png = sorted(figures - png_stems)
    if missing_png:
        preview = ", ".join(missing_png[:3])
        more = "..." if len(missing_png) > 3 else ""
        return False, f"missing PNGs for {preview}{more}"

    if expected_sigma_coupling is not None:
        if "sigma_coupling" not in summary.columns:
            return False, "summary has no sigma_coupling column"
        coupling_values = set(summary["sigma_coupling"].dropna().astype(str).unique())
        if coupling_values != {expected_sigma_coupling}:
            return False, f"sigma_coupling is {sorted(coupling_values)}, expected {expected_sigma_coupling}"

    return True, f"{len(figures)} figures and {len(summary)} rows are ready"


def run_or_load(section, runner, output_dir, force=False, expected_sigma_coupling=None, **runner_kwargs):
    ready, reason = output_is_ready(output_dir, expected_sigma_coupling=expected_sigma_coupling)
    if force or not ready:
        action = "regenerating" if force else "generating"
        display(Markdown(f"**{section}**: {action} outputs. Reason: `{reason}`"))
        paths, summary_df = runner(output_dir=output_dir, **runner_kwargs)
    else:
        display(Markdown(f"**{section}**: reusing existing outputs. `{reason}`"))
        summary_df = pd.read_csv(output_dir / "case_summary.csv")
        paths = sorted(output_dir.glob("*.png")) + [output_dir / "case_summary.csv"]

    display(
        pd.DataFrame(
            [
                {
                    "section": section,
                    "output_dir": str(output_dir),
                    "paths": len(paths),
                    "summary_rows": len(summary_df),
                    "figures": summary_df["figure"].nunique() if "figure" in summary_df else None,
                }
            ]
        )
    )
    display(summary_df.head())
    return paths, summary_df


def select_pngs(output_dir, patterns=None, limit=None):
    paths = sorted(output_dir.glob("*.png"))
    if patterns:
        paths = [path for path in paths if any(fnmatch.fnmatch(path.name, pattern) for pattern in patterns)]
    if limit is not None:
        paths = paths[:limit]
    return paths


def display_png_gallery(section, output_dir, patterns=None, limit=None, width=1200):
    all_pngs = sorted(output_dir.glob("*.png"))
    paths = select_pngs(output_dir, patterns=patterns, limit=limit)
    display(Markdown(f"### {section} figures"))
    display(Markdown(f"Showing `{len(paths)}` of `{len(all_pngs)}` PNG files from `{output_dir}`."))
    if not paths:
        display(Markdown("No PNG files matched the current display patterns."))
        return
    for path in paths:
        display(Markdown(f"`{path.name}`"))
        display(Image(filename=str(path), width=width))


def display_summary_png_gallery(section, output_dir, summary_df, patterns=None, limit=None, width=1200):
    if summary_df is None or "figure" not in summary_df:
        display_png_gallery(section, output_dir, patterns=patterns, limit=limit, width=width)
        return

    figure_names = sorted(summary_df["figure"].dropna().astype(str).unique())
    paths = [output_dir / f"{figure}.png" for figure in figure_names]
    paths = [path for path in paths if path.exists()]
    if patterns:
        paths = [path for path in paths if any(fnmatch.fnmatch(path.name, pattern) for pattern in patterns)]
    if limit is not None:
        paths = paths[:limit]

    display(Markdown(f"### {section} figures"))
    display(Markdown(f"Showing `{len(paths)}` current-summary PNG files from `{output_dir}`."))
    if not paths:
        display(Markdown("No PNG files matched the current summary and display patterns."))
        return
    for path in paths:
        display(Markdown(f"`{path.name}`"))
        display(Image(filename=str(path), width=width))

## Step1

In [ ]:
step1_spec_kwargs = {
    "reference_lam_mu_pairs": STEP1_REFERENCE_LAM_MU_PAIRS,
    "a_values": STEP1_A_VALUES,
    "b_ratios": STEP1_B_RATIOS,
    "mu_sweeps": STEP1_MU_SWEEPS,
    "lam_sweeps": STEP1_LAM_SWEEPS,
}
step1_specs = make_step1_specs(**step1_spec_kwargs)
step1_spec_table = specs_table("step1", step1_specs)
display(
    pd.DataFrame(
        [
            {
                "section": "step1",
                "figures": len(step1_specs),
                "parameter_combos": int(step1_spec_table["n_combos"].sum()),
                "output_dir": str(STEP1_OUTPUT_DIR),
            }
        ]
    )
)
display(step1_spec_table.head())

if RUN_STEP1:
    step1_paths, step1_summary = run_or_load(
        "step1",
        run_step1_experiments,
        STEP1_OUTPUT_DIR,
        force=FORCE_RERUN,
        **step1_spec_kwargs,
    )
else:
    display(Markdown("**step1** is disabled by `RUN_STEP1=False`."))

In [ ]:
if RUN_STEP1:
    display_png_gallery(
        "step1",
        STEP1_OUTPUT_DIR,
        patterns=STEP1_DISPLAY_PATTERNS,
        limit=DISPLAY_LIMIT,
        width=DISPLAY_WIDTH,
    )

## Step2

In [ ]:
step2_spec_kwargs = {
    "reference_rho_kappa_pairs": STEP2_REFERENCE_RHO_KAPPA_PAIRS,
    "sigma_a_values": STEP2_SIGMA_A_VALUES,
    "sigma_b_ratios": STEP2_SIGMA_B_RATIOS,
    "kappa_sweeps": STEP2_KAPPA_SWEEPS,
    "rho_sweeps": STEP2_RHO_SWEEPS,
}
step2_specs = make_step2_specs(sigma_coupling=STEP2_SIGMA_COUPLING, **step2_spec_kwargs)
step2_spec_table = specs_table("step2", step2_specs)
display(
    pd.DataFrame(
        [
            {
                "section": "step2",
                "sigma_coupling": STEP2_SIGMA_COUPLING,
                "figures": len(step2_specs),
                "parameter_combos": int(step2_spec_table["n_combos"].sum()),
                "output_dir": str(STEP2_OUTPUT_DIR),
            }
        ]
    )
)
display(step2_spec_table.head())

if RUN_STEP2:
    step2_paths, step2_summary = run_or_load(
        "step2",
        run_step2_experiments,
        STEP2_OUTPUT_DIR,
        force=FORCE_RERUN,
        expected_sigma_coupling=STEP2_SIGMA_COUPLING,
        sigma_coupling=STEP2_SIGMA_COUPLING,
        **step2_spec_kwargs,
    )
else:
    display(Markdown("**step2** is disabled by `RUN_STEP2=False`."))

In [ ]:
if RUN_STEP2:
    display_png_gallery(
        "step2",
        STEP2_OUTPUT_DIR,
        patterns=STEP2_DISPLAY_PATTERNS,
        limit=DISPLAY_LIMIT,
        width=DISPLAY_WIDTH,
    )

## Step2s

In [ ]:
step2s_spec_kwargs = {
    "reference_rho_kappa_pairs": STEP2S_REFERENCE_RHO_KAPPA_PAIRS,
    "sigma_a_values": STEP2S_SIGMA_A_VALUES,
    "sigma_b_ratios": STEP2S_SIGMA_B_RATIOS,
    "kappa_sweeps": STEP2S_KAPPA_SWEEPS,
    "rho_sweeps": STEP2S_RHO_SWEEPS,
}
step2s_specs = make_step2_specs(sigma_coupling=STEP2S_SIGMA_COUPLING, **step2s_spec_kwargs)
step2s_spec_table = specs_table("step2s", step2s_specs)
display(
    pd.DataFrame(
        [
            {
                "section": "step2s",
                "sigma_coupling": STEP2S_SIGMA_COUPLING,
                "figures": len(step2s_specs),
                "parameter_combos": int(step2s_spec_table["n_combos"].sum()),
                "output_dir": str(STEP2S_OUTPUT_DIR),
            }
        ]
    )
)
display(step2s_spec_table.head())

if RUN_STEP2S:
    step2s_paths, step2s_summary = run_or_load(
        "step2s",
        run_step2_experiments,
        STEP2S_OUTPUT_DIR,
        force=FORCE_RERUN,
        expected_sigma_coupling=STEP2S_SIGMA_COUPLING,
        sigma_coupling=STEP2S_SIGMA_COUPLING,
        **step2s_spec_kwargs,
    )
else:
    display(Markdown("**step2s** is disabled by `RUN_STEP2S=False`."))

In [ ]:
if RUN_STEP2S:
    display_png_gallery(
        "step2s",
        STEP2S_OUTPUT_DIR,
        patterns=STEP2S_DISPLAY_PATTERNS,
        limit=DISPLAY_LIMIT,
        width=DISPLAY_WIDTH,
    )

## Kuramoto

In [ ]:
kuramoto_spec_kwargs = {
    "case1_fixed_k_intra": KURAMOTO_CASE1_FIXED_K_INTRA,
    "case1_k_inter_values": KURAMOTO_CASE1_K_INTER_VALUES,
    "case2_fixed_k_inter": KURAMOTO_CASE2_FIXED_K_INTER,
    "case2_k_intra_values": KURAMOTO_CASE2_K_INTRA_VALUES,
}
kuramoto_specs = make_kuramoto_specs(**kuramoto_spec_kwargs)
kuramoto_spec_table = specs_table("kuramoto", kuramoto_specs)
display(
    pd.DataFrame(
        [
            {
                "section": "kuramoto",
                "N": KURAMOTO_CONFIG.N,
                "n_clusters": KURAMOTO_CONFIG.n_clusters,
                "noise": KURAMOTO_CONFIG.noise,
                "figures": len(kuramoto_specs),
                "parameter_combos": int(kuramoto_spec_table["n_combos"].sum()),
                "output_dir": str(KURAMOTO_OUTPUT_DIR),
            }
        ]
    )
)
display(kuramoto_spec_table)

if RUN_KURAMOTO:
    kuramoto_paths, kuramoto_summary = run_or_load(
        "kuramoto",
        run_kuramoto_experiments,
        KURAMOTO_OUTPUT_DIR,
        force=FORCE_RERUN,
        config=KURAMOTO_CONFIG,
        **kuramoto_spec_kwargs,
    )
else:
    display(Markdown("**kuramoto** is disabled by `RUN_KURAMOTO=False`."))

In [ ]:
if RUN_KURAMOTO:
    display_summary_png_gallery(
        "kuramoto",
        KURAMOTO_OUTPUT_DIR,
        kuramoto_summary,
        patterns=KURAMOTO_DISPLAY_PATTERNS,
        limit=DISPLAY_LIMIT,
        width=DISPLAY_WIDTH,
    )

## Combined Summary

In [ ]:
summary_frames = []
for section_name, variable_name in (
    ("step1", "step1_summary"),
    ("step2", "step2_summary"),
    ("step2s", "step2s_summary"),
    ("kuramoto", "kuramoto_summary"),
):
    if variable_name in globals():
        frame = globals()[variable_name].copy()
        frame.insert(0, "section", section_name)
        summary_frames.append(frame)

if summary_frames:
    combined_summary = pd.concat(summary_frames, ignore_index=True, sort=False)
    display(
        combined_summary.groupby("section", dropna=False)
        .agg(rows=("combo", "count"), figures=("figure", "nunique"))
        .reset_index()
    )
    display(combined_summary.head())
else:
    display(Markdown("No section summaries are loaded. Enable at least one run switch above."))